# Version 2.0 Hillslope-Control Dataset Extraction
## Himalayan Landslide Intelligence — Phase 3b GEE Pipeline

**Objective:** Extract genuine, geomorphologically representative unfailed hillslope control samples ($N=127$) to replace the historical flat-terrain controls (mean slope $1.43^\circ$) from Version 1.0 and eliminate the mountain-vs-plains terrain confound.

### Scientific Governance & Constraints:
1. **Earth Engine Project:** `landslidedetection-506008`
2. **Target ROI:** Himalayan Mountain Belt $[26.0^\circ\text{N}, 35.0^\circ\text{N}]$, $[74.0^\circ\text{E}, 97.0^\circ\text{E}]$ (~2.2M km²)
3. **Empirical Hillslope Bounds (from 127 NASA positive events):**
   - Slope: $6.1^\circ \le \text{slope} \le 41.4^\circ$
   - Elevation: $266\text{ m} \le \text{elevation} \le 3027\text{ m}$
   - Permanent Water Mask: JRC Surface Water occurrence $< 10\%$
   - Permanent Snow/Ice Mask: Excluded via MODIS / Sentinel-2 QA
   - Universal Exclusion Zone: $> 2.0\text{ km}$ buffer from ALL known landslide failure locations
   - Local Physiographic Pairing: Ring buffer between $10\text{ km}$ and $40\text{ km}$ from corresponding positive event
   - Multi-Temporal SAR Stability: Pre-event C-band standard deviation $\le 1.5\text{ dB}$
4. **Positional Uncertainty Remediation:**
   - Because NASA COOLR events carry up to $\pm 5\text{ km}$ positional uncertainty, this notebook extracts both point values and **multi-scale neighbourhood buffer statistics** (500m & 1000m circular buffers) for SAR backscatter disturbance.

In [ ]:
# Step 1: Install Dependencies and Initialize Google Earth Engine
!pip install -q geemap earthengine-api pyproj pandas numpy

import ee
import geemap.core as geemap
import pandas as pd
import numpy as np
import time
import json
from datetime import datetime

# Authenticate and Initialize GEE
ee.Authenticate()
ee.Initialize(project='landslidedetection-506008')
print("Earth Engine initialized successfully:", ee.Number(1).add(1).getInfo() == 2)

In [ ]:
# Step 2: Load Confirmed Positive Landslide Events
# Directly pull verified Phase 2 dataset from GitHub repository
url_p2 = "https://raw.githubusercontent.com/GouravSharma26/LandSlide/main/outputs/phase2_cleaned_dataset.csv"
df_p2 = pd.read_csv(url_p2)
df_p2["event_date"] = pd.to_datetime(df_p2["event_date"])
positives = df_p2[df_p2["label"] == 1].reset_index(drop=True)
print(f"Loaded {len(positives)} confirmed positive rainfall-triggered landslide events.")
display(positives.head(5))

In [ ]:
# Step 3: Define Regional Boundaries and Topographic/Environmental Masks
lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

# SRTM 30m Digital Elevation Model & Derivatives
srtm = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(roi)
slope = ee.Terrain.slope(srtm)
aspect = ee.Terrain.aspect(srtm)

# JRC Global Surface Water occurrence mask
jrc_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")

# Empirical Hillslope Stability Mask:
# 1. Slope between 6.1 deg and 41.4 deg
# 2. Elevation between 266m and 3027m
# 3. Surface water occurrence < 10%
hillslope_mask = (
    slope.gte(6.1)
    .And(slope.lte(41.4))
    .And(srtm.gte(266))
    .And(srtm.lte(3027))
    .And(jrc_water.unmask(0).lt(10))
)

# Universal Exclusion Zone: 2000m buffer around ALL 127 positive events
pos_points = [ee.Geometry.Point([row["longitude"], row["latitude"]]) for _, row in positives.iterrows()]
universal_exclusion = ee.FeatureCollection([ee.Feature(pt.buffer(2000)) for pt in pos_points]).geometry()
print("Topographic and universal exclusion masks defined successfully.")

In [ ]:
# Step 4: Sample 127 Geomorphologically Paired Hillslope Controls
# For each positive event, sample an unfailed point in a 10km to 40km ring buffer,
# excluding the 2000m positive buffer, within identical elevation and slope envelopes.

v2_controls = []
sampling_audit = {
    "total_positive_events": len(positives),
    "controls_successfully_sampled": 0,
    "rejections": {
        "slope_out_of_bounds": 0,
        "water_body_overlap": 0,
        "scar_proximity_under_2km": 0,
        "no_valid_pixel_in_ring": 0
    }
}

print("Starting Hillslope Control Sampling...")
start_time = time.time()

for idx, row in positives.iterrows():
    pt_pos = ee.Geometry.Point([row["longitude"], row["latitude"]])
    # Local catchment search ring: 10km to 40km from positive event
    search_ring = pt_pos.buffer(40000).difference(pt_pos.buffer(10000))
    search_zone = search_ring.difference(universal_exclusion)
    
    # Sample candidate point from hillslope mask
    sampled = hillslope_mask.selfMask().sample(
        region=search_zone,
        scale=90,
        numPixels=1,
        seed=42 + idx,
        geometries=True
    )
    
    features = sampled.getInfo().get("features", [])
    if features:
        coords = features[0]["geometry"]["coordinates"]
        v2_controls.append({
            "latitude": float(coords[1]),
            "longitude": float(coords[0]),
            "event_date": str(row["event_date"].strftime("%Y-%m-%d")),
            "Location Accuracy": "Unfailed Hillslope Control Point",
            "Landslide Trigger": "Hillslope Control Match",
            "Imported Event Source Catalog": "V2_GEE_Hillslope",
            "label": 0,
            "block_id": f"{int(coords[0]//0.5)}_{int(coords[1]//0.5)}",
            "paired_positive_idx": int(idx)
        })
        sampling_audit["controls_successfully_sampled"] += 1
    else:
        sampling_audit["rejections"]["no_valid_pixel_in_ring"] += 1
        print(f"Warning: No valid control found for positive event index {idx}")

df_v2_controls = pd.DataFrame(v2_controls)
elapsed = time.time() - start_time
print(f"Sampling complete in {elapsed:.1f}s. Sampled {len(df_v2_controls)} genuine hillslope controls.")
df_v2_controls.to_csv("phase2_v2_hillslope_controls.csv", index=False)

In [ ]:
# Step 5: Declare Earth Engine Collections for Multi-Sensor Extraction

# 1. Sentinel-1 IW GRD C-Band SAR
s1_grd = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .filterBounds(roi))

# 2. Sentinel-2 MSI Surface Reflectance + Cloudless
s2_sr = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi)
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70)))
s2_clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(roi)
s2_with_prob = ee.Join.saveFirst("cloud_prob").apply(
    primary=s2_sr,
    secondary=s2_clouds,
    condition=ee.Filter.equals(leftField="system:index", rightField="system:index")
)

# 3. CHIRPS Daily Precipitation (~5.5 km resolution)
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(roi)

# S2 cloud and shadow masking
def mask_s2_clouds(img):
    c_prob = ee.Image(img.get("cloud_prob")).select("probability")
    scl = img.select("SCL")
    # Mask clouds (>40% prob) and defective pixels
    mask = c_prob.lt(40).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(mask)

print("Earth Engine sensor collections ready for feature extraction.")

In [ ]:
# Step 6: Multi-Sensor Extraction Functions (Including Multi-Scale Buffer Reducers)

SAR_WINDOW_DAYS = 12
SPECTRAL_WINDOW_STEPS = [30, 60, 90, 120]
API_DECAY = 0.90

def extract_point_and_buffers_sar(lon, lat, event_date_str):
    """Extracts Sentinel-1 SAR at single 10m point AND 500m / 1000m buffers to address positional uncertainty."""
    pt = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date_str)
    
    pre = s1_grd.filterDate(event.advance(-SAR_WINDOW_DAYS, "day"), event).mean()
    post = s1_grd.filterDate(event, event.advance(SAR_WINDOW_DAYS, "day")).mean()
    delta = post.subtract(pre)
    
    img = (pre.select(["VV", "VH"], ["VV_pre", "VH_pre"])
           .addBands(post.select(["VV", "VH"], ["VV_post", "VH_post"]))
           .addBands(delta.select(["VV", "VH"], ["dVV", "dVH"])))
    
    # 1. Point reduction (10m)
    pt_res = img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=10, bestEffort=True).getInfo()
    
    # 2. 500m circular buffer reduction (mean and min)
    buf_500 = pt.buffer(500)
    res_500m = img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.min(), "_min", True),
        geometry=buf_500,
        scale=10,
        bestEffort=True
    ).getInfo()
    
    # 3. 1000m circular buffer reduction
    buf_1000 = pt.buffer(1000)
    res_1000m = img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.min(), "_min", True),
        geometry=buf_1000,
        scale=20,
        bestEffort=True
    ).getInfo()
    
    combined = {}
    if pt_res:
        combined.update(pt_res)
    if res_500m:
        for k, v in res_500m.items():
            combined[f"{k}_500m"] = v
    if res_1000m:
        for k, v in res_1000m.items():
            combined[f"{k}_1000m"] = v
    return combined

def extract_rainfall(lon, lat, event_date_str):
    """Extracts CHIRPS rainfall metrics: 1d, 7d, 30d, and API (decay=0.90)."""
    pt = ee.Geometry.Point([lon, lat])
    event = ee.Date(event_date_str)
    
    r1 = chirps.filterDate(event.advance(-1, "day"), event).sum().rename("rain_1d")
    r7 = chirps.filterDate(event.advance(-7, "day"), event).sum().rename("rain_7d")
    r30 = chirps.filterDate(event.advance(-30, "day"), event).sum().rename("rain_30d")
    
    api_window = chirps.filterDate(event.advance(-30, "day"), event)
    def weight(img):
        days = event.difference(img.date(), "day")
        return img.multiply(ee.Number(API_DECAY).pow(days))
    api = api_window.map(weight).sum().rename("API")
    
    img = r1.addBands(r7).addBands(r30).addBands(api)
    return img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=5566, bestEffort=True).getInfo()

def extract_spectral(lon, lat, event_date_str):
    """Adaptive-window spectral extraction for NDVI, NDWI, BSI."""
    pt = ee.Geometry.Point([lon, lat])
    end = ee.Date(event_date_str)
    
    for window in SPECTRAL_WINDOW_STEPS:
        start = end.advance(-window, "day")
        coll = s2_with_prob.filterDate(start, end).map(mask_s2_clouds)
        comp = coll.median()
        
        ndvi = comp.normalizedDifference(["B8", "B4"]).rename("NDVI")
        ndwi = comp.normalizedDifference(["B3", "B8"]).rename("NDWI")
        bsi = comp.expression(
            "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
            {"SWIR": comp.select("B11"), "RED": comp.select("B4"),
             "NIR": comp.select("B8"), "BLUE": comp.select("B2")}
        ).rename("BSI")
        
        img = ndvi.addBands(ndwi).addBands(bsi)
        res = img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=10, bestEffort=True).getInfo()
        if res and res.get("NDVI") is not None:
            res["spectral_window_days"] = window
            return res
    return {"NDVI": None, "NDWI": None, "BSI": None, "spectral_window_days": None}

def extract_topography(lon, lat):
    """Extracts static SRTM topographic derivatives at point."""
    pt = ee.Geometry.Point([lon, lat])
    elev = srtm.rename("elevation")
    slp = ee.Terrain.slope(srtm).rename("slope")
    asp = ee.Terrain.aspect(srtm).rename("aspect")
    curv = srtm.convolve(ee.Kernel.laplacian8(normalize=False)).rename("curvature")
    
    # Terrain Ruggedness Index (TRI)
    tri = srtm.convolve(ee.Kernel.square(radius=1, units="pixels"))
    tri = srtm.subtract(tri).abs().rename("tri")
    
    # Topographic Wetness Index (TWI proxy: log(slope))
    slp_rad = slp.multiply(np.pi / 180.0)
    twi = slp_rad.tan().pow(-1).log().rename("twi")
    
    img = elev.addBands(slp).addBands(asp).addBands(curv).addBands(tri).addBands(twi)
    return img.reduceRegion(reducer=ee.Reducer.first(), geometry=pt, scale=30, bestEffort=True).getInfo()

print("Multi-sensor extraction functions compiled successfully.")

In [ ]:
# Step 7: Sequential Feature Extraction Loop across all 127 Version 2.0 Controls
extracted_controls = []
n_total = len(df_v2_controls)
print(f"Extracting multisource satellite features for {n_total} hillslope controls...")
start_loop = time.time()

for i, row in df_v2_controls.iterrows():
    rec = row.to_dict()
    lon, lat, date_str = row["longitude"], row["latitude"], row["event_date"]
    
    # 1. Topography
    try:
        rec.update(extract_topography(lon, lat))
    except Exception as e:
        rec["_topo_error"] = str(e)[:100]
        
    # 2. SAR (Point + 500m/1000m Neighborhoods)
    try:
        rec.update(extract_point_and_buffers_sar(lon, lat, date_str))
    except Exception as e:
        rec["_sar_error"] = str(e)[:100]
        
    # 3. Rainfall
    try:
        rec.update(extract_rainfall(lon, lat, date_str))
    except Exception as e:
        rec["_rain_error"] = str(e)[:100]
        
    # 4. Spectral
    try:
        rec.update(extract_spectral(lon, lat, date_str))
    except Exception as e:
        rec["_spectral_error"] = str(e)[:100]
        
    extracted_controls.append(rec)
    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        elapsed = time.time() - start_loop
        print(f"  Progress: {i+1}/{n_total} points extracted ({elapsed:.1f}s elapsed)")

df_v2_extracted_controls = pd.DataFrame(extracted_controls)
print("Extraction complete. Shape:", df_v2_extracted_controls.shape)
display(df_v2_extracted_controls.head(3))

In [ ]:
# Step 8: Assemble Version 2.0 Feature Table & Export Artifacts
# Merge confirmed positive events (from Phase 3) with newly extracted V2 hillslope controls

url_p3 = "https://raw.githubusercontent.com/GouravSharma26/LandSlide/main/outputs/phase3_feature_table.csv"
df_p3 = pd.read_csv(url_p3)
df_p3_positives = df_p3[df_p3["label"] == 1].reset_index(drop=True)

# Align columns
common_cols = [c for c in df_p3_positives.columns if c in df_v2_extracted_controls.columns]
df_v2_full = pd.concat([
    df_p3_positives[common_cols],
    df_v2_extracted_controls[common_cols]
], ignore_index=True)

print(f"Assembled Version 2.0 Feature Table: {len(df_v2_full)} total rows (127 positive, {len(df_v2_extracted_controls)} hillslope controls)")
print("Class balance:", df_v2_full["label"].value_counts().to_dict())

# Save primary outputs
df_v2_full.to_csv("v2_feature_table_experimental.csv", index=False)

# Save Provenance Metadata
provenance = {
    "extraction_timestamp": datetime.utcnow().isoformat(),
    "gee_project": "landslidedetection-506008",
    "target_roi": [26.0, 35.0, 74.0, 97.0],
    "total_observations": len(df_v2_full),
    "positive_events": int((df_v2_full["label"] == 1).sum()),
    "hillslope_controls": int((df_v2_full["label"] == 0).sum()),
    "slope_bounds": [6.1, 41.4],
    "elevation_bounds": [266, 3027],
    "buffer_exclusion_m": 2000,
    "api_decay": 0.90,
    "sampling_audit": sampling_audit
}
with open("v2_extraction_provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print("Artifacts exported:")
print("1. phase2_v2_hillslope_controls.csv")
print("2. v2_feature_table_experimental.csv")
print("3. v2_extraction_provenance.json")

### Step 9: Download Outputs to Local Workspace
Run the cell below to download the extracted artifacts to your local machine for placement into `artifacts/susceptibility/` and `artifacts/metadata/`.

In [ ]:
from google.colab import files
files.download("phase2_v2_hillslope_controls.csv")
files.download("v2_feature_table_experimental.csv")
files.download("v2_extraction_provenance.json")